<img src="https://images.squarespace-cdn.com/content/v1/67351b6c456f5a62d1c1bca2/17a93d03-0cae-49bf-b152-8dfd0cdb9d7d/Quantum+Rings+Logo+Main+White.png?format=300w" alt="Quantum Rings" width="150" align="right"/>

# The Bernstein–Vazirani Algorithm

The Bernstein–Vazirani algorithm was invented by Ethan Bernstein and Umesh Vazirani in 1997.

This sample code implements Bernstein–Vazirani, following the explanation in the Quantum Rings documentation on the Bernstein–Vazirani algorithm here:
https://portal.quantumrings.com/doc/BernsteinVazirani.html

## Required One-time Setup

If you haven't already done so, you will need to obtain a free license key from the Quantum Rings portal. Replace the `token` and `name` placeholders below with your key and the email address associated with your account.

In [ ]:
%%capture
%pip install QuantumRingsLib matplotlib numpy

In [ ]:
import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
provider = QuantumRingsProvider(token="YOUR_TOKEN_HERE", name="YOUR_EMAIL_ADDRESS")
backend = provider.get_backend("scarlet_quantum_rings")
shots = 100

provider.active_account()

In [ ]:
def bv_func (qc, hiddenstring):

    # set the ancilla qubit to state |1>
    qc.x(qc.num_qubits-1)

    # Set all qubits to a superposition state.
    for i in range (qc.num_qubits):
        qc.h(i)

    qc.barrier()

    # implement the hidden string - the oracle function
    for i in range (qc.num_qubits - 1):
        if ( 0!= ( hiddenstring & (1 << i))):
            qc.cx(i, qc.num_qubits-1)

    qc.barrier()

    # Apply H gates one more time
    for i in range (qc.num_qubits-1):
        qc.h(i)

    qc.barrier()

    # finally measure
    for i in range(qc.num_qubits - 1):
        qc.measure(i,i)

    return

def plot_histogram (counts, title=""):
    """
    Plots the histogram of the counts

    Args:

        counts (dict):
            The dictionary containing the counts of states

        titles (str):
            A title for the graph.

    Returns:
        None

    """
    fig, ax = plt.subplots(figsize =(10, 7))
    plt.xlabel("States")
    plt.ylabel("Counts")
    mylist = [key for key, val in counts.items() for _ in range(val)]

    unique, inverse = np.unique(mylist, return_inverse=True)
    bin_counts = np.bincount(inverse)

    plt.bar(unique, bin_counts)

    maxFreq = max(counts.values())
    plt.ylim(ymax=np.ceil(maxFreq / 10) * 10 if maxFreq % 10 else maxFreq + 10)
    # Show plot
    plt.title(title)
    plt.show()
    return

In [ ]:
# obtain the hidden string from the user
hiddenstring = int(input("Enter the hidden string as a binary value: "), 2)

#determine the number of qubits required to represent the hidden string
numberofqubits = hiddenstring.bit_length()

print(hiddenstring, numberofqubits)

# add a qubit for the ancilla register
numberofqubits += 1

q = QuantumRegister(numberofqubits , 'q')
c = ClassicalRegister(numberofqubits , 'c')
qc = QuantumCircuit(q, c)

bv_func(qc, hiddenstring)

job = backend.run(qc, shots=shots)
job_monitor(job)
result = job.result()
counts = result.get_counts()

plot_histogram(counts,"")